# Microgrid Co-Simulation

This notebook demonstrates a **multi-energy** (electrical + thermal) co-simulation
combining models with different time scales:

| Simulator | Type | Step Size | Time Scale |
|---|---|---|---|
| Battery | Dynamic (Euler) | 10 s | Fast |
| Heat pump | Algebraic | 30 s | Fast |
| Thermal storage | Dynamic (Euler) | 60 s | Medium |
| Greenhouse | Dynamic (Euler) | 60 s | Medium |
| Controller | Rule-based | 900 s | Slow |
| Power grid | Steady-state PF | 300 s | Slow |
| Weather data | CSV playback | 900 s | Slow |

The macro time-step is **900 s** (15 min).  energysim automatically
sub-steps each simulator at its own micro time-step within every macro
interval, exchanging variables at the macro boundary.

In [ ]:
import os, sys
from pathlib import Path

HERE = Path(os.getcwd()).resolve()
# Ensure energysim is importable from the dev source tree
_src = str(HERE.parents[1] / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from energysim import world
print('energysim imported successfully')

In [ ]:
# Generate data files if they don't exist
if not (HERE / 'weather_data.csv').exists() or not (HERE / 'microgrid.p').exists():
    import generate_data
    generate_data.main()
    print('Data files generated.')
else:
    print('Data files already exist.')

## Create Co-Simulation World

We register seven simulators — each with its own step size.

In [ ]:
SIM_DIR = str(HERE / 'simulators')

my_world = world(start_time=0, stop_time=86400, logging=True, t_macro=900)

# CSV weather / demand profiles
my_world.add_simulator(
    sim_type='csv', sim_name='weather',
    sim_loc=str(HERE / 'weather_data.csv'),
    outputs=['T_ambient', 'solar_irradiance', 'elec_demand',
             'heat_demand', 'elec_price', 'pv_power'],
    step_size=900)

# Pandapower microgrid
my_world.add_simulator(
    sim_type='powerflow', sim_name='grid',
    sim_loc=str(HERE / 'microgrid.p'),
    inputs=['PV.p_mw', 'Battery.p_mw', 'HeatPump.p_mw', 'House.p_mw'],
    outputs=['Bus0.vm_pu', 'Bus1.vm_pu', 'Bus2.vm_pu',
             'PV.p_mw', 'Battery.p_mw', 'HeatPump.p_mw', 'House.p_mw'],
    step_size=300, pf='pf')

# Battery — fast dynamics  (10 s)
my_world.add_simulator(
    sim_type='external', sim_name='battery', sim_loc=SIM_DIR,
    inputs=['P_cmd'], outputs=['SoC', 'P_actual'], step_size=10)

# Heat pump — fast dynamics  (30 s)
my_world.add_simulator(
    sim_type='external', sim_name='heatpump', sim_loc=SIM_DIR,
    inputs=['P_cmd', 'T_source', 'T_sink'],
    outputs=['Q_thermal', 'COP', 'P_elec'], step_size=30)

# Thermal storage — medium dynamics  (60 s)
my_world.add_simulator(
    sim_type='external', sim_name='thermal_tank', sim_loc=SIM_DIR,
    inputs=['Q_in', 'Q_out', 'T_ambient'],
    outputs=['T_storage', 'Q_loss'], step_size=60)

# Greenhouse — medium dynamics  (60 s)
my_world.add_simulator(
    sim_type='external', sim_name='greenhouse', sim_loc=SIM_DIR,
    inputs=['Q_heating', 'T_ambient', 'solar_irradiance'],
    outputs=['T_inside', 'Q_demand'], step_size=60)

# Energy management controller — slow  (900 s)
my_world.add_simulator(
    sim_type='external', sim_name='controller', sim_loc=SIM_DIR,
    inputs=['SoC', 'T_storage', 'T_greenhouse', 'P_pv',
            'elec_price', 'heat_demand_signal', 'T_ambient'],
    outputs=['P_battery_cmd', 'P_hp_cmd'], step_size=900)

## Define Connections

The connections dictionary maps **output** variables to **input** variables.
Fan-out (one output → multiple inputs) uses **tuples**.

In [ ]:
connections = {
    # Weather -> various
    'weather.T_ambient':        ('heatpump.T_source', 'thermal_tank.T_ambient',
                                 'greenhouse.T_ambient', 'controller.T_ambient'),
    'weather.solar_irradiance': 'greenhouse.solar_irradiance',
    'weather.pv_power':         ('grid.PV.p_mw', 'controller.P_pv'),
    'weather.elec_demand':      'grid.House.p_mw',
    'weather.heat_demand':      'controller.heat_demand_signal',
    'weather.elec_price':       'controller.elec_price',
    # Controller -> actuators
    'controller.P_battery_cmd': 'battery.P_cmd',
    'controller.P_hp_cmd':      'heatpump.P_cmd',
    # Battery -> grid + controller
    'battery.P_actual':         'grid.Battery.p_mw',
    'battery.SoC':              'controller.SoC',
    # Heat pump -> thermal chain + grid
    'heatpump.Q_thermal':       'thermal_tank.Q_in',
    'heatpump.P_elec':          'grid.HeatPump.p_mw',
    # Thermal tank -> feedback
    'thermal_tank.T_storage':   ('heatpump.T_sink', 'controller.T_storage'),
    # Greenhouse <-> thermal
    'greenhouse.Q_demand':      'thermal_tank.Q_out',
    'greenhouse.T_inside':      'controller.T_greenhouse',
}
my_world.add_connections(connections)

## Run Simulation

In [ ]:
my_world.simulate(pbar=True, record_all=False)

## Analyse Results

In [ ]:
import matplotlib.pyplot as plt

results = my_world.results(
    to_csv=False, dashboard=True,
    dashboard_path=str(HERE / 'microgrid_dashboard.html'))

# Extract DataFrames
df_bat  = results['battery']
df_hp   = results['heatpump']
df_tank = results['thermal_tank']
df_gh   = results['greenhouse']
df_ctrl = results['controller']
df_grid = results['grid']
df_wx   = results['weather']

fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

# 1. Battery SoC
ax = axes[0]
ax.plot(df_bat['time'] / 3600, df_bat['SoC'], 'b-', lw=1.5)
ax.set_ylabel('SoC [-]')
ax.set_title('Battery State of Charge')
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)

# 2. Temperatures
ax = axes[1]
ax.plot(df_gh['time'] / 3600,   df_gh['T_inside'],    'r-',  label='Greenhouse')
ax.plot(df_tank['time'] / 3600, df_tank['T_storage'],  'm-',  label='Thermal tank')
ax.plot(df_wx['time'] / 3600,   df_wx['T_ambient'],    'c--', label='Ambient')
ax.axhline(18, color='gray', ls=':', label='T_set_low (18 \u00b0C)')
ax.axhline(22, color='gray', ls='--', label='T_set_high (22 \u00b0C)')
ax.set_ylabel('Temperature [\u00b0C]')
ax.set_title('Temperatures')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)

# 3. Power flows
ax = axes[2]
ax.plot(df_wx['time'] / 3600,  df_wx['pv_power'] * 1000,   'orange', label='PV')
ax.plot(df_bat['time'] / 3600, df_bat['P_actual'] * 1000,  'b-',     label='Battery')
ax.plot(df_hp['time'] / 3600,  df_hp['P_elec'] * 1000,     'r-',     label='Heat pump')
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('Power [kW]')
ax.set_title('Power Flows')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)

# 4. Controller outputs + price
ax = axes[3]
ax2 = ax.twinx()
ax.plot(df_ctrl['time'] / 3600, df_ctrl['P_battery_cmd'] * 1000, 'b-', label='Batt cmd')
ax.plot(df_ctrl['time'] / 3600, df_ctrl['P_hp_cmd'] * 1000,      'r-', label='HP cmd')
ax2.plot(df_wx['time'] / 3600,  df_wx['elec_price'], 'g--', alpha=0.6, label='Price')
ax.set_ylabel('Power cmd [kW]')
ax2.set_ylabel('Price [EUR/kWh]')
ax.set_xlabel('Time [h]')
ax.set_title('Controller Outputs & Electricity Price')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='best', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('microgrid_results.png', dpi=150)
plt.show()

## Key Observations

- The **battery** charges during low-price overnight periods and from excess PV around midday.
- The **heat pump** maintains the greenhouse temperature within the 18–22 °C band
  using proportional control, modulated by thermal-tank temperature.
- The **thermal storage** smooths the heat supply to the greenhouse.
- **Different time scales** (10 s to 900 s) are handled seamlessly by energysim —
  each simulator runs at its own micro time-step within the shared macro interval.